# S6E11 — model comparison

The readable companion to `python -m scripts.run_experiment`. It calls the
same `src.models` registry and the same `src.cv.run_cv`, so a number produced
here is reproducible from the command line and vice versa.

Every result is 5-fold **stratified** CV. Stratification is not optional at a
1.67% positive rate: unstratified folds would vary in prevalence enough to
move AUC on its own.

In [ ]:
# Notebooks live in notebooks/; the package lives at the repo root.
# Importing from src/ rather than re-implementing keeps the notebook and the
# scripts from drifting apart -- there is exactly one definition of the data
# handling, the models and the CV loop.
import sys, pathlib
ROOT = pathlib.Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt

from src import config
from src.data import load_raw, split_xy

pd.set_option("display.width", 120)
print("repo root:", ROOT)

from src.cv import run_cv
from src.models import REGISTRY
from src.submit import write_submission
print("registered models:", ", ".join(sorted(REGISTRY)))

In [ ]:
train, test = load_raw()
X, y, X_test, test_ids = split_xy(train, test)
print(f"X {X.shape}   positives {int(y.sum()):,} ({y.mean():.4f})")

# Guard against the two mistakes that silently invalidate everything.
assert config.TARGET_COL not in X.columns, "target leaked into features"
assert config.ID_COL not in X.columns, "id leaked into features"
assert list(X.columns) == list(X_test.columns), "train/test feature mismatch"
print("no id/target leakage; train and test columns match")

## The model set, and why each one is here

| model | role |
|---|---|
| `linear_main` | the control — logistic regression, main effects only |
| `linear_inter` | tests whether feature crosses buy anything |
| `tree_d4` | the legible model; prints as a diagram, sets a floor |
| `nb_binned` | tests how far independent marginal evidence alone gets |
| `hgb` | scikit-learn gradient boosting, handles categoricals natively |
| `lgbm` | the tuning vehicle |

Each one answers a question. None is there because it is popular.

In [ ]:
import time
def evaluate(name, n_rows=0, folds=config.N_FOLDS, seed=config.SEED):
    """Fit one registered model under stratified CV and return a summary row."""
    Xs, ys = (X, y)
    if n_rows:
        Xs = X.sample(n=n_rows, random_state=seed)
        ys = y.loc[Xs.index]
    t0 = time.time()
    res = run_cv(Xs, ys, X_test, model_name=name, n_folds=folds, verbose=False)
    return {"model": name, "n_rows": len(Xs), "folds": folds,
            "mean_auc": float(np.mean(res.fold_scores)),
            "std_auc": float(np.std(res.fold_scores)),
            "seconds": round(time.time() - t0, 1)}, res

### The cheap models first

`nb_binned` and `tree_d4` run in seconds and bound the problem from below.

In [ ]:
rows = []
for name in ["nb_binned", "tree_d4", "linear_main"]:
    row, _ = evaluate(name)
    rows.append(row); print(row)

### Gradient boosting

In [ ]:
row_hgb, res_hgb = evaluate("hgb")
rows.append(row_hgb); row_hgb

## The interaction test

The question: do all pairwise feature crosses add anything over main effects?

Two things make this experiment easy to get wrong:

1. **It must be run against a matched control.** `linear_inter` on a subsample
   compared against `linear_main` on full data measures the subsample, not the
   interactions.
2. **It will not fit in memory on full data.** 25 features expand to 1,225
   polynomial terms; at 560k training rows that is a 5.1 GiB dense array. The
   honest move is to subsample *both* arms and say so, not to quietly drop the
   experiment.

In [ ]:
SUB = 150_000
inter_rows = []
for name in ["linear_main", "linear_inter"]:
    row, _ = evaluate(name, n_rows=SUB)
    inter_rows.append(row); print(row)

a, b = inter_rows[0]["mean_auc"], inter_rows[1]["mean_auc"]
spread = max(r["std_auc"] for r in inter_rows)
print(f"\ninteractions change AUC by {b-a:+.5f}")
print(f"fold-to-fold std is {spread:.5f}")
print("verdict:", "inside the noise" if abs(b-a) < spread else "outside the noise")

## Results

In [ ]:
results = pd.DataFrame(rows + inter_rows).sort_values("mean_auc", ascending=False)
results.round({"mean_auc": 5, "std_auc": 5})

In [ ]:
n_full = len(X)
full = results.query("n_rows == @n_full").sort_values("mean_auc")
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.barh(full["model"], full["mean_auc"], xerr=full["std_auc"],
        color="#4C72B0", ecolor="#333", capsize=3)
ax.set_xlim(0.75, 0.96)
ax.set_xlabel("mean fold ROC AUC  (5-fold stratified, full data)")
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()

## Submission

`write_submission` checks the shape and id alignment before writing, so a
malformed file fails here rather than on Kaggle.

In [ ]:
path = write_submission(test_ids, res_hgb.test_pred, tag="hgb_notebook")
sub = pd.read_csv(path)
print(path)
print(sub.shape, list(sub.columns))
print(f"predicted mean {sub[config.TARGET_COL].mean():.5f} vs train rate {y.mean():.5f}")
sub.head()

Upload with:

```bash
kaggle competitions submit -c playground-series-s6e11 -f <path> -m "hgb baseline"
```

Then record the leaderboard score against the CV number. The gap between them
is itself a result worth reporting.